### Aarjeyan Shrestha - AML 3304
#### Changes and Experiments in the chatbot_learning.ipynb files

Feb 24, 2025

#### `1.  Changes in Activation Functions`
- **Change:**  
  - **Original:** Sigmoid activation was used for all layers.
  - **New:**  
    - **Hidden Layers:** Continue using sigmoid activation.
    - **Output Layer:** Switched to softmax activation.
- **Advantages:**  
  - **Softmax on Output:** Normalizes outputs into a probability distribution (sums to 1), which is ideal for multi-class classification.
  - **Maintaining Sigmoid in Hidden Layers:** Provides the necessary non-linearity to model complex patterns.


In [1]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def softmax(z):
    # Subtract max for numerical stability along the correct axis (assumes z is (n, 1))
    z_shift = z - np.max(z, axis=0, keepdims=True)
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)


#### `2. Changes in the Loss Function, Optimization Method`

In [2]:
class NeuralNetwork:
    def __init__(self, layer_sizes, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        """
        layer_sizes: List of neuron counts for each layer. (e.g., [input_size, hidden_size, output_size])
        learning_rate: Step size for updates.
        beta1, beta2, epsilon: Adam optimizer hyperparameters.
        """
        self.num_layers = len(layer_sizes)
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.adam_t = 0  # Time step for Adam
        
        self.weights = []
        self.biases = []
        # Adam moment estimates for weights and biases
        self.m_w = []
        self.v_w = []
        self.m_b = []
        self.v_b = []
        
        # Initialize weights, biases and Adam variables
        for i in range(self.num_layers - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i])
            b = np.random.randn(layer_sizes[i+1], 1)
            self.weights.append(w)
            self.biases.append(b)
            
            # Initialize Adam moments with zeros (same shape as w and b)
            self.m_w.append(np.zeros_like(w))
            self.v_w.append(np.zeros_like(w))
            self.m_b.append(np.zeros_like(b))
            self.v_b.append(np.zeros_like(b))
    
    def forward(self, x):
        """
        Performs a forward pass through the network.
        Hidden layers: sigmoid activation.
        Output layer: softmax activation.
        Returns:
          activations: List of activations (including input and output).
          zs: List of z vectors (weighted sums) for each layer.
        """
        activation = x
        activations = [x]
        zs = []
        for i, (w, b) in enumerate(zip(self.weights, self.biases)):
            z = np.dot(w, activation) + b
            zs.append(z)
            if i == len(self.weights) - 1:
                activation = softmax(z)
            else:
                activation = sigmoid(z)
            activations.append(activation)
        return activations, zs

    def compute_loss(self, output, target):
        """
        Cross-entropy loss for a single example.
        """
        epsilon = 1e-12
        output = np.clip(output, epsilon, 1. - epsilon)
        return -np.sum(target * np.log(output))

    def backward(self, activations, zs, target):
        """
        Backpropagation to compute gradients.
        For the output layer with softmax & cross-entropy, the gradient is (output - target).
        """
        grad_w = [np.zeros_like(w) for w in self.weights]
        grad_b = [np.zeros_like(b) for b in self.biases]
        
        # Output layer error
        delta = (activations[-1] - target)  # shape: (output_size, 1)
        grad_w[-1] = np.dot(delta, activations[-2].T)
        grad_b[-1] = delta
        
        # Backpropagate error for hidden layers
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_derivative(z)
            delta = np.dot(self.weights[-l+1].T, delta) * sp
            grad_w[-l] = np.dot(delta, activations[-l-1].T)
            grad_b[-l] = delta
            
        return grad_w, grad_b

    def update_parameters(self, grad_w, grad_b):
        """
        Updates network parameters using the Adam optimizer.
        """
        self.adam_t += 1  # Increment timestep
        
        for i in range(len(self.weights)):
            # Update moments for weights
            self.m_w[i] = self.beta1 * self.m_w[i] + (1 - self.beta1) * grad_w[i]
            self.v_w[i] = self.beta2 * self.v_w[i] + (1 - self.beta2) * (grad_w[i] ** 2)
            
            # Bias-corrected moments for weights
            m_hat_w = self.m_w[i] / (1 - self.beta1 ** self.adam_t)
            v_hat_w = self.v_w[i] / (1 - self.beta2 ** self.adam_t)
            
            # Update moments for biases
            self.m_b[i] = self.beta1 * self.m_b[i] + (1 - self.beta1) * grad_b[i]
            self.v_b[i] = self.beta2 * self.v_b[i] + (1 - self.beta2) * (grad_b[i] ** 2)
            
            # Bias-corrected moments for biases
            m_hat_b = self.m_b[i] / (1 - self.beta1 ** self.adam_t)
            v_hat_b = self.v_b[i] / (1 - self.beta2 ** self.adam_t)
            
            # Update weights and biases
            self.weights[i] -= self.learning_rate * m_hat_w / (np.sqrt(v_hat_w) + self.epsilon)
            self.biases[i]  -= self.learning_rate * m_hat_b / (np.sqrt(v_hat_b) + self.epsilon)

    def train(self, x, target):
        """
        Train on a single example.
        Returns the loss for that example.
        """
        activations, zs = self.forward(x)
        loss = self.compute_loss(activations[-1], target)
        grad_w, grad_b = self.backward(activations, zs, target)
        self.update_parameters(grad_w, grad_b)
        return loss

    def train_batch(self, batch_data):
        """
        Train on a mini-batch by accumulating gradients.
        Returns the average loss over the batch.
        """
        batch_size = len(batch_data)
        # Initialize accumulators for gradients with zeros
        accum_grad_w = [np.zeros_like(w) for w in self.weights]
        accum_grad_b = [np.zeros_like(b) for b in self.biases]
        total_loss = 0
        
        for x, target in batch_data:
            activations, zs = self.forward(x)
            loss = self.compute_loss(activations[-1], target)
            total_loss += loss
            grad_w, grad_b = self.backward(activations, zs, target)
            accum_grad_w = [acc + gw for acc, gw in zip(accum_grad_w, grad_w)]
            accum_grad_b = [acc + gb for acc, gb in zip(accum_grad_b, grad_b)]
        
        # Average gradients
        accum_grad_w = [gw / batch_size for gw in accum_grad_w]
        accum_grad_b = [gb / batch_size for gb in accum_grad_b]
        # Update parameters using the averaged gradients
        self.update_parameters(accum_grad_w, accum_grad_b)
        return total_loss / batch_size


#### `2. Changes in Loss Function`
- **Change:**  
  - **Original:** Mean Squared Error (MSE) loss.
  - **New:** Cross-Entropy loss.
- **Advantages:**  
  - **Cross-Entropy:** Better measures the difference between predicted and true probability distributions, leading to more informative gradients and improved convergence for classification tasks.

#### `3. Changes in Optimization Method`
- **Change:**  
  - **Original:** Basic gradient descent or per-example updates.
  - **New:** Adam optimizer.
- **Advantages:**  
  - **Adam:** Automatically adapts learning rates for individual parameters and incorporates momentum, often resulting in faster convergence and enhanced training performance.

#### `Extended Intents Dataset`
- **Change:**  
  - **Original:** A very limited set of intents and patterns.
  - **New:** Expanded dataset with more diverse patterns and additional intents (e.g., "info", "joke", "help", "weather").
- **Advantages:**  
  - **Dataset Expansion:** Improves the chatbot's ability to generalize, resulting in more accurate intent classification and more appropriate responses.


### `Preparing Training Data`

In [3]:

intents = {
    "greeting": {
         "patterns": [
             "Hi", "Hello", "Hey", "Good morning",
             "What's up?", "How's it going?"
         ],
         "responses": [
             "Hello!", "Hi there!", "Greetings!",
             "Hey! How can I help you?"
         ]
    },
    "farewell": {
         "patterns": [
             "Bye", "See you", "Goodbye", "Later",
             "Catch you later", "Talk to you later"
         ],
         "responses": [
             "Goodbye!", "See you later!", "Farewell!",
             "Bye! Have a great day!"
         ]
    },
    "thanks": {
         "patterns": [
             "Thanks", "Thank you", "Much appreciated",
             "Thanks a lot", "Thank you so much"
         ],
         "responses": [
             "You're welcome!", "No problem!",
             "Anytime!", "Happy to help!"
         ]
    },
    "info": {
         "patterns": [
             "What is your name?", "Who are you?",
             "Identify yourself", "Tell me about you"
         ],
         "responses": [
             "I am your friendly chatbot.",
             "I'm a chatbot designed to help you!",
             "I'm a chatbot powered by AI."
         ]
    },
    "joke": {
         "patterns": [
             "Tell me a joke", "I need a laugh",
             "Make me laugh", "Do you know any jokes?"
         ],
         "responses": [
             "Why did the scarecrow win an award? Because he was outstanding in his field!",
             "I would tell you a construction joke, but I'm still working on it."
         ]
    },
    "help": {
         "patterns": [
             "Help", "I need help", "Can you help me?",
             "Assistance"
         ],
         "responses": [
             "How can I help you?",
             "Sure, what do you need assistance with?",
             "I'm here to help!"
         ]
    },
    "weather": {
         "patterns": [
             "What's the weather like?", "Tell me the weather",
             "How is the weather today?", "Is it sunny?"
         ],
         "responses": [
             "I'm not sure, but I hope it's nice!",
             "Check your local weather forecast.",
             "I can't predict the weather yet."
         ]
    }
}


In [4]:
corpus = []
for intent_data in intents.values():
    corpus.extend(intent_data["patterns"])

print("Corpus:")
for sentence in corpus:
    print(" -", sentence)


Corpus:
 - Hi
 - Hello
 - Hey
 - Good morning
 - What's up?
 - How's it going?
 - Bye
 - See you
 - Goodbye
 - Later
 - Catch you later
 - Talk to you later
 - Thanks
 - Thank you
 - Much appreciated
 - Thanks a lot
 - Thank you so much
 - What is your name?
 - Who are you?
 - Identify yourself
 - Tell me about you
 - Tell me a joke
 - I need a laugh
 - Make me laugh
 - Do you know any jokes?
 - Help
 - I need help
 - Can you help me?
 - Assistance
 - What's the weather like?
 - Tell me the weather
 - How is the weather today?
 - Is it sunny?


#### `4. Changes in Feature Representation`
- **Change:**  
  - **Original:** Basic bag-of-words approach.
  - **New:** TF-IDF vectorization or Word2Vec (using TF-IDF, but can be changed in classify_intent function).
- **Advantages:**  
  - **TF-IDF:** Weighs words based on their importance, reducing the impact of common, less-informative words, and yields more discriminative features for text classification.

#### `TF-IDF`

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit the TF-IDF vectorizer on our corpus 
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

def sentence_to_tfidf(sentence):
    # Transform returns a row vector; transpose to get a column vector.
    return vectorizer.transform([sentence]).T

# Create mapping for intents to indices 
intents_list = list(intents.keys())
intent_to_index = {intent: i for i, intent in enumerate(intents_list)}

def intent_to_onehot(intent):
    onehot = np.zeros(len(intent_to_index))
    onehot[intent_to_index[intent]] = 1
    return onehot.reshape(-1, 1)

# Build training data using TF-IDF features
training_data_tfidf = []
for intent, data in intents.items():
    for pattern in data["patterns"]:
        # sentence_to_tfidf already returns a column vector, so we do not transpose here
        vec = sentence_to_tfidf(pattern).toarray()  # Shape should be (n_features, 1)
        label = intent_to_onehot(intent)
        training_data_tfidf.append((vec, label))

print("Training examples (TF-IDF):", len(training_data_tfidf))


Training examples (TF-IDF): 33


#### `Word2Vec`

In [6]:
#!pip install gensim nltk -q

import nltk
nltk.download('punkt', quiet=True)
from gensim.models import Word2Vec
import re

def tokenize(sentence):
    return re.findall(r'\w+', sentence.lower())

# Tokenize each sentence in the corpus
tokenized_patterns = [tokenize(sentence) for sentence in corpus]

# Train Word2Vec model on our corpus
word2vec_model = Word2Vec(sentences=tokenized_patterns, vector_size=50, window=5, min_count=1, workers=4)

def sentence_to_word2vec(sentence, model, oov_vector=None):
    tokens = tokenize(sentence)
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
        else:
            if oov_vector is None:
                oov_vector = np.zeros(model.vector_size)
            vectors.append(oov_vector)
    if len(vectors) == 0:
        return np.zeros((model.vector_size, 1))
    avg_vector = np.mean(vectors, axis=0)
    return avg_vector.reshape(-1, 1)

# Build training data using Word2Vec embeddings
training_data_word2vec = []
for intent, data in intents.items():
    for pattern in data["patterns"]:
        vec = sentence_to_word2vec(pattern, word2vec_model)
        label = intent_to_onehot(intent)
        training_data_word2vec.append((vec, label))

print("Number of training examples (Word2Vec):", len(training_data_word2vec))


Number of training examples (Word2Vec): 33


#### `Defining network architecture and Training`

In [7]:
# Determine input and output sizes based on TF-IDF features and intents.
input_size = len(vectorizer.get_feature_names_out())  # e.g., 50 features
hidden_size = 8      # You can adjust this value
output_size = len(intents_list)

# Create an instance of the neural network.
chatbot_nn = NeuralNetwork(layer_sizes=[input_size, hidden_size, output_size],
                           learning_rate=0.001)  

# Training parameters
epochs = 10000
loss_history = []

for epoch in range(epochs):
    loss = chatbot_nn.train_batch(training_data_tfidf)
    loss_history.append(loss)
    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss:.4f}")


Epoch 1000: Loss = 0.8313
Epoch 2000: Loss = 0.2047
Epoch 3000: Loss = 0.0685
Epoch 4000: Loss = 0.0281
Epoch 5000: Loss = 0.0133
Epoch 6000: Loss = 0.0069
Epoch 7000: Loss = 0.0038
Epoch 8000: Loss = 0.0022
Epoch 9000: Loss = 0.0013
Epoch 10000: Loss = 0.0007


### `Chatbot Interface`

In [8]:

import random

def classify_intent(sentence, nn, vectorizer, intents_list, intent_to_index):
    """
    Convert the input sentence to a TF-IDF column vector, run it through the network,
    and return the predicted intent along with the network's output probabilities.
    """
    # Convert sentence to TF-IDF vector; no extra transpose is needed.
    tfidf_vec = sentence_to_tfidf(sentence).toarray()  # shape: (n_features, 1)
    activations, _ = nn.forward(tfidf_vec)
    output = activations[-1]
    intent_index = np.argmax(output)
    return intents_list[intent_index], output

def chat():
    print("Chatbot is online! Type 'quit' to exit.")
    while True:
        user_input = input("You: ")
        print(f"You: {user_input} ")
        if user_input.lower() == "quit":
            print("Chatbot: Goodbye!")
            break
        
        intent, output_probs = classify_intent(user_input, chatbot_nn, vectorizer, intents_list, intent_to_index)
        responses = intents[intent]["responses"]
        response = random.choice(responses)
        print(f"Chatbot ({intent}): {response}")

# chat function
chat()


Chatbot is online! Type 'quit' to exit.
You: Hi 
Chatbot (greeting): Hi there!
You: Good Morning 
Chatbot (greeting): Greetings!
You: who are you? 
Chatbot (info): I am your friendly chatbot.
You: Who are you? 
Chatbot (info): I'm a chatbot designed to help you!
You: Tell me a joke 
Chatbot (joke): I would tell you a construction joke, but I'm still working on it.
You: I need help 
Chatbot (help): How can I help you?
You: Is it sunny? 
Chatbot (weather): I can't predict the weather yet.
You: Ok thanks a lot 
Chatbot (thanks): You're welcome!
You: later 
Chatbot (farewell): Bye! Have a great day!
You: quit 
Chatbot: Goodbye!
